# 28. 数据读取与保存

<!-- module-learning-arc:start -->
> **Pandas 模块主线｜第 7 / 10 步：让分析结果可以保存与复现**
>
> **持续应用背景：** 搭建电商履约异常追踪台：把订单、客户、商品和履约信息整理成安全合并的事实表，再生成趋势指标和异常工单。
>
> **承接上一阶段：** 文本、日期与特征处理  →  **本章任务：** 数据读取与保存  →  **下一步：** 分组、聚合与数据透视
>
> **大作业连接：** 本章练习将成为《电商履约异常追踪台》的一部分，最终需要从多表质量审计走到订单粒度事实表、窗口趋势和可复核异常工单。
<!-- module-learning-arc:end -->


## 本章场景

真实数据很少天生就住在 Python 里，多半是以 CSV、JSON 这些文件存在硬盘上。



## 本章目标

学完本章，你将能够：

- **理解**：理解常见数据源（csv/excel/json）读写与 dtype。
- **操作**：能用 pandas 读取/保存常用格式并核对列类型。
- **迁移**：能把本地文件读入并另存为分析所需的干净数据集。


## 28.1 核心概念

**背景引入**：真实数据很少天生就住在 Python 里，多半是以 CSV、JSON 这些文件存在硬盘上。能不能把文件读对、把类型和缺失值理清、再把结果干净地导出，决定了你每次分析的起点顺不顺——商品编号前导零被吞掉、保存多出一列行号，都是这一环节最常翻的跟头。先把“进出文件”这一关打通，后面的分组、绘图、建模才不会被原始数据问题反复绊倒。

- 读取时明确分隔符、编码、日期和类型。
- 保存索引前先判断它是否是业务字段。
- 浏览器课程使用内存文本模拟文件，方法与真实路径一致。

> **直观类比**：读取文件就像把硬盘上的表格逐格誊进一本新账，Pandas 会“猜”每列的类型——编号 007 会被猜成数字 7，前导零就丢了；用 dtype="string" 就是明说“这一列是编号，按文本留”，别让机器自作主张。


## 28.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| read_csv() | `pd.read_csv()` | StringIO可以在没有真实文件的情况下练习CSV读取。 | 让商品编号被自动读取为整数并丢失前导零 |
| read_csv() 控制分隔符和类型 | `pd.read_csv()` | 读取时明确sep和dtype，可以避免编号前导零丢失；str写法兼容旧版Pandas。 | 保存默认索引造成Unnamed列 |
| to_numeric() | `pd.Series()`、`pd.to_numeric()` | 先转换，再检查新产生的缺失值。 | 未检查日期和数值解析失败 |
| to_csv() | `pd.DataFrame()`、`data.to_csv()` | index=False可以避免把默认行号写成多余列。 | 让商品编号被自动读取为整数并丢失前导零 |
| to_json() | `pd.DataFrame()`、`data.to_json()` | orient='records'适合把每一行导出为一个JSON对象。 | 保存默认索引造成Unnamed列 |


## 28.3 示例 1：读取CSV文本

**背景引入**：真实数据很少直接躺在 Python 里，多半是 CSV 这类文本文件。老板丢来一个 "orders.csv"，第一步就是把它正确读进来——表头、分隔符、日期列都要对，读错了后面全是白忙。

**讲解**：用 `StringIO` 把内存里的文本变得像文件一样，喂给 `pd.read_csv` 就能练习读取；`parse_dates` 让日期列直接变成日期类型。

- `StringIO(csv_text)` 让字符串像文件对象一样可读，浏览器里练 CSV 全靠它；
- `pd.read_csv(...)` 用逗号自动切分，第一行自动当表头；
- `parse_dates=["date"]` 指定列在读入时转成日期类型，`print(orders.dtypes)` 验证；
- amount 列那行空值会读成 NaN，正好衔接后面的缺失处理；
- **口诀**：StringIO 假扮文件，read_csv 读入表，日期列记得 parse_dates。


In [ ]:
from io import StringIO
import pandas as pd

csv_text = """order_id,region,amount,date
A1,华东,320.5,2026-01-05
A2,华南,880.0,2026-01-08
A3,华北,,2026-01-12
"""
orders = pd.read_csv(StringIO(csv_text), parse_dates=["date"])
print(orders)
print(orders.dtypes)


## 28.4 示例 2：控制缺失和类型

**背景引入**：读文件最常见的两个坑：商品编号 "001" 被自动当成整数变成 1，前导零丢了；库存里的 "N/A" 被当成普通文本，等你想算合计时才发现它不是数字。根源都在读取那一刻没把类型和缺失值"说清楚"。

**讲解**：在 `read_csv` 时用 `dtype` 指定每列类型、用 `na_values` 声明哪些写法算缺失，读取结果一次到位。

- `dtype={"product_id": "string", "category": "category"}`：编号按文本存，保住前导零；
- `na_values=["N/A"]`：把 "N/A" 声明为缺失值，读进来直接就是 NaN；
- 用 `print(products.dtypes)` 复核：category 列的类型、stock 是否数值型；
- **口诀**：编号要保前导零，dtype 指定文本型；N/A 想变缺失值，na_values 说一声。


In [ ]:
csv_text = """product_id,category,stock
001,办公,18
002,数码,N/A
003,家居,24
"""
products = pd.read_csv(
    StringIO(csv_text),
    dtype={"product_id": "string", "category": "category"},
    na_values=["N/A"],
)
print(products)
print(products.dtypes)


## 28.5 示例 3：导出CSV与JSON

**背景引入**：分析做完不是终点，结果得交出去——导成 CSV 给同事用 Excel 打开，或转成 JSON 给系统接口消费。这两步看着简单，最容易翻车的就是**多导出的一列行号**（Unnamed），还有中文和日期格式不对。

**讲解**：`to_csv(index=False)` 导出干净 CSV、不要默认行号列；`to_json(orient="records")` 把每行变成一个 JSON 对象。

- `index=False`：不把默认行号写进 CSV，避免生成多余的 Unnamed 列；
- `to_json(orient="records", force_ascii=False, date_format="iso")`：每行一个对象、中文不乱码、日期用 ISO 形式；
- 用 `print` 直接查看导出文本，确认表头、内容、格式都符合预期；
- **口诀**：导 CSV 记得 index=False，导 JSON 用 records，中文 ISO 一步到位。


In [ ]:
output_csv = orders.to_csv(index=False)
output_json = orders.to_json(
    orient="records", force_ascii=False, date_format="iso"
)
print(output_csv)
print(output_json)


## 28.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# read_csv()
# StringIO可以在没有真实文件的情况下练习CSV读取。
from io import StringIO
import pandas as pd

text = "id,amount\nA1,320\nA2,880\n"
data = pd.read_csv(StringIO(text))
print(data)


In [ ]:
# read_csv() 控制分隔符和类型
# 读取时明确sep和dtype，可以避免编号前导零丢失；str写法兼容旧版Pandas。
from io import StringIO
import pandas as pd

text = "product_id;stock\n001;18\n002;24\n"
data = pd.read_csv(StringIO(text), sep=";", dtype={"product_id": str})
print(data)
print(data.dtypes)


In [ ]:
# to_numeric()
# 先转换，再检查新产生的缺失值。
import pandas as pd

raw = pd.Series(["320", "invalid", "880"])
numbers = pd.to_numeric(raw, errors="coerce")
print(numbers)


In [ ]:
# to_csv()
# index=False可以避免把默认行号写成多余列。
import pandas as pd

data = pd.DataFrame({"id": ["A1", "A2"], "amount": [320, 880]})
print(data.to_csv(index=False))


In [ ]:
# to_json()
# orient='records'适合把每一行导出为一个JSON对象。
import pandas as pd

data = pd.DataFrame({"id": ["A1", "A2"], "amount": [320, 880]})
print(data.to_json(orient="records", force_ascii=False))


**练一练 24.6**：处理一张以分号分隔的商品表，编号含前导零，数量列混入无效值。完成三件事：① 用 `read_csv` 读取文本，指定 `sep=";"`，并把 `product_id` 读成字符串以保住前导零；② 用 `to_numeric(errors="coerce")` 把 `quantity` 转成数值，无效值变成缺失值；③ 删除 `quantity` 缺失的行，再用 `to_csv(index=False)` 导出并打印。数据就用简单的数值表即可。


In [ ]:
# 请在下方填写代码
from io import StringIO
import pandas as pd

text = """product_id;category;quantity
001;办公;18
002;数码;未知
003;家居;24
"""

# TODO ①: 用 read_csv 读取 text，sep=";"，把 product_id 指定为字符串类型，存到 inventory

# TODO ②: 把 inventory["quantity"] 用 to_numeric(errors="coerce") 转为数值

# TODO ③: 删除 quantity 缺失的行，存到 inventory_clean，并打印
# inventory_clean.to_csv(index=False)


## 28.7 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Description": "string",
        "Country": "category",
    },
).rename(
    columns={
        "InvoiceNo": "order_id",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "order_time",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }
)
large_orders["sales"] = (
    large_orders["quantity"] * large_orders["unit_price"]
).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C")
    | (large_orders["quantity"] < 0),
    "取消/退货",
    "完成",
)
print("UCI Online Retail 公开数据：")
print(f"  {len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print(
    "内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)
large_orders.head()


In [ ]:
loaded = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "Country": "category",
    },
)
print(f"从公开 CSV 读取：{loaded.shape}，日期类型：{loaded['InvoiceDate'].dtype}")
display(loaded.head())


## 28.8 独立迁移练习

替换一个字段或分组口径，并核对处理前后的行数与粒度。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 28.9 本章实训：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame(
    {
        "region": ["华东", "华东", "华南", "华南"],
        "channel": ["线上", "线下", "线上", "线下"],
        "sales": [120, 80, 150, 100],
    }
)
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 28.9.1 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。



In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 28.9.2 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。



## 28.10 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 28.10.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。



## 28.11 易错点提醒

- 让商品编号被自动读取为整数并丢失前导零
- 保存默认索引造成Unnamed列
- 未检查日期和数值解析失败


## 28.12 练习与作业

1. 读取一段分号分隔文本
2. 指定日期列
3. 筛选有效记录并导出CSV

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 28.13 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“读取一段分号分隔文本”。
2. **独立完成**：不复制示例代码，完成“指定日期列”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“筛选有效记录并导出CSV”，用一两句话说明你修改了什么。

### 28.13.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 28.13.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
from io import StringIO
import pandas as pd

text = """date;region;sales
2026-01-01;华东;120
2026-01-02;华南;98
2026-01-03;华北;invalid
"""
# TODO: 读取CSV文本，使用分号分隔符，解析日期列
# TODO: 转换sales列为数值，无效值转为缺失值
# TODO: 删除sales列缺失的行
# TODO：请在下方完成 —— 24.13 练习与作业 1. 读取一段分号分隔文本 2. 指定日期列 3. 筛选有效记录并导出CSV 提交前检查：代码可


In [ ]:
from io import StringIO
import pandas as pd

text = """date;region;sales
2026-01-01;华东;120
2026-01-02;华南;98
2026-01-03;华北;invalid
"""
data = pd.read_csv(StringIO(text), sep=";", parse_dates=["date"])
data["sales"] = pd.to_numeric(data["sales"], errors="coerce")
clean = data.dropna(subset=["sales"])
print(clean.to_csv(index=False))


## 28.14 小结

掌握CSV与JSON的读取、参数控制、内存往返和结果导出。

**迁移思考**：

1. 如果 CSV 文件中的日期格式是"2026/01/05"而不是"2026-01-05"，pd.to_datetime 还能自动识别吗？
2. 为什么商品编号要用 dtype="string" 而不是让 Pandas 自动推断类型？



### 28.14.1 你已经掌握

- 从文本读取表格
- 控制类型和缺失值
- 处理编码与日期
- 导出可复用结果



### 28.14.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。



### 28.14.3 需要注意

- 让商品编号被自动读取为整数并丢失前导零
- 保存默认索引造成Unnamed列
- 未检查日期和数值解析失败



### 28.14.4 完成检查

- [ ] 能够从文本读取表格
- [ ] 能够控制类型和缺失值
- [ ] 能够处理编码与日期
- [ ] 能够导出可复用结果



### 28.14.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。

